### This is the cleaning and preparation of summary statistics

In [42]:
import pandas as pd

Parameters include
1. Grouping results by 
State
Year
Month

2. All of the US

3. Demographics
Filter to: 
Single-Year Ages < 1 for all infants
Infant Age Groups <1 day to 7-27 days
Single Race 6

4. All years and months

5. All weekday, autopsy, place of death

6. All underlying cause of death

7. All MCD

8. Export to CSV
Don't show totals
Show Zero Values
Show Suppressed Values

In [43]:
#Import Raw Data - Already Implemented
raw = pd.read_csv(r"Data\Raw\infant_raw_state.csv")

raw_neonatal = pd.read_csv(r"Data\Raw\neonatal_raw_state.csv")

raw_births = pd.read_csv(r"Data\Raw\births_raw.csv")

raw_births_college = pd.read_csv('Data/Raw/births_college_educated_raw.csv')
raw_births_35 = pd.read_csv('Data/Raw/births_over_35_raw.csv')
raw_births_medicaid = pd.read_csv('Data/Raw/births_medicaid_raw.csv')


In [44]:
renaming_dict = {'State':'state',
                 'State of Residence':'state',
                 'State Code':'state_code',
                 'State of Residence Code':'state_code',
                 'Year':'year',
                 'Month':'month',
                 'Births':'births',
                'Deaths':'deaths',
                'Month Code':'month_code'
                 }

def clean_month(month_str):
    if pd.isnull(month_str):
        return 0
    
    if 'Jan' in month_str:
        return 1
    elif 'Feb' in month_str:
        return 2
    elif 'Mar' in month_str:
        return 3
    elif 'Apr' in month_str:
        return 4
    elif 'May' in month_str:
        return 5
    elif 'Jun' in month_str:
        return 6
    elif 'Jul' in month_str:
        return 7
    elif 'Aug' in month_str:
        return 8
    elif 'Sep' in month_str:
        return 9
    elif 'Oct' in month_str:
        return 10
    elif 'Nov' in month_str:
        return 11
    elif 'Dec' in month_str:
        return 12
    else:
        return 0

#Cleaning Data
def clean_infant_data(raw_df):
    # Clean the raw data
    df = raw_df.copy(deep = True)
    df['Deaths'] = df['Deaths'].replace('Suppressed', 0).fillna(0).astype(int)
    df = df.rename(columns=renaming_dict)
    df['month'] = df['month'].apply(clean_month)

    df['state_code'] = df['state_code'].astype(str).str.replace('.0','',regex=False).str.zfill(2)
    df = df[['state','state_code','year','month','deaths']]
    return df

def clean_births_data(raw_df):
    # Clean the raw data
    df = raw_df.copy(deep = True)
    df = df.rename(columns=renaming_dict)

    df = df.drop(columns = ['month'])
    df['month'] = df['month_code']

    df['state_code'] = df['state_code'].astype(str).str.replace('.0','',regex = False).str.zfill(2)
    df = df[['state','state_code','year','month','births']]
    return df

In [45]:
#Clean
df = clean_infant_data(raw)

df_neonatal = clean_infant_data(raw_neonatal).rename(columns = {'deaths':'deaths_neonatal'})

df_births = clean_births_data(raw_births)

df_births_35 = clean_births_data(raw_births_35).rename(columns = {'births':'births_35'})

df_births_college = clean_births_data(raw_births_college).rename(columns = {'births':'births_college'})

df_births_medicaid = clean_births_data(raw_births_medicaid).rename(columns = {'births':'births_medicaid'})

#Limit to deaths > 10
#df = df[df['deaths'] > 10]

#df_neonatal = df_neonatal[df_neonatal['deaths_neonatal'] > 10]



In [46]:
#Merge together
df_merged = lambda x, y: pd.merge(x, y, on=['state_code', 'year', 'month'], how='outer')
dataframes = [df, df_neonatal, df_births, df_births_35,
              df_births_college, df_births_medicaid]

df_merged = dataframes[0]
for next_df in dataframes[1:]:
    df_merged = pd.merge(
        df_merged,
        next_df,
        on=['state','state_code', 'year', 'month'],
        how='left'
    )

In [47]:
#Merge
df_merged
df_merged = df_merged.drop_duplicates()

In [48]:
df_merged

,state,state_code,year,month,deaths,deaths_neonatal,births,births_35,births_college,births_medicaid
0,Alabama,01,2018.0,1,40,21,4902.0,581.0,1128.0,2431.0
1,Alabama,01,2018.0,2,41,28,4342.0,514.0,992.0,2176.0
2,Alabama,01,2018.0,3,31,19,4761.0,597.0,1215.0,2326.0
3,Alabama,01,2018.0,4,32,22,4393.0,530.0,1091.0,2165.0
4,Alabama,01,2018.0,5,32,16,4785.0,520.0,1163.0,2284.0
...,...,...,...,...,...,...,...,...,...,...
4280,Wyoming,56,2024.0,9,0,0,483.0,66.0,147.0,159.0
4281,Wyoming,56,2024.0,10,0,0,526.0,88.0,160.0,145.0
4282,Wyoming,56,2024.0,11,0,0,445.0,65.0,141.0,113.0
4283,Wyoming,56,2024.0,12,0,0,494.0,66.0,142.0,161.0


In [49]:
#Export Clean Versions
df_merged.to_csv('Data/Clean/infant_birth_death.csv', index=False)

In [50]:
#Calculate Rate
df_merged['rate']  = (df_merged['deaths'] / df_merged['births']) * 1000
df_merged['rate_neonatal']  = (df_merged['deaths_neonatal'] / df_merged['births']) * 1000


In [51]:
print(df_merged.head())

     state state_code    year  month  deaths  deaths_neonatal  births  \
0  Alabama         01  2018.0      1      40               21  4902.0   
1  Alabama         01  2018.0      2      41               28  4342.0   
2  Alabama         01  2018.0      3      31               19  4761.0   
3  Alabama         01  2018.0      4      32               22  4393.0   
4  Alabama         01  2018.0      5      32               16  4785.0   

   births_35  births_college  births_medicaid      rate  rate_neonatal  
0      581.0          1128.0           2431.0  8.159935       4.283966  
1      514.0           992.0           2176.0  9.442653       6.448641  
2      597.0          1215.0           2326.0  6.511237       3.990758  
3      530.0          1091.0           2165.0  7.284316       5.007967  
4      520.0          1163.0           2284.0  6.687565       3.343783  


In [54]:
#Reproducing Table 1
def get_table_1(df,neonatal = True, tx = True):
    # Group by Year and State, summing Deaths
    #if neonatal
    if neonatal == True:
        sum_col = 'deaths_neonatal'
    else: 
        sum_col = 'deaths'
        
    a = df.groupby(['year','state'])[[sum_col,'births']].sum().reset_index()
    # Pivot for Texas
    if tx == True:
        out = a[a['state'] == 'Texas'].groupby('year')[[sum_col,'births']].sum().reset_index()
    else: 
        out = a[a['state'] != 'Texas'].groupby('year')[[sum_col,'births']].sum().reset_index()
    out['rate'] = out[sum_col] / out['births'] * 1000
    out['change_yoy_deaths'] = out[sum_col].diff() / out[sum_col].shift(1) * 100
    out['change_yoy_rate'] = out['rate'].diff() / out['rate'].shift(1) * 100

    return out


#INFANT
table_1a,table_1b = get_table_1(df_merged,False,True),get_table_1(df_merged,False,False)
print(table_1a)
#print(table_1b)


#NEONATAL
table_1c,table_1d = get_table_1(df_merged,True,True),get_table_1(df_merged,True,False)
#print(table_1c)
#print(table_1d)


     year  deaths    births      rate  change_yoy_deaths  change_yoy_rate
0  2018.0    2083  378624.0  5.501500                NaN              NaN
1  2019.0    2075  377599.0  5.495248          -0.384061        -0.113652
2  2020.0    1968  368190.0  5.345066          -5.156627        -2.732929
3  2021.0    1985  373594.0  5.313254           0.863821        -0.595164
4  2022.0    2243  389741.0  5.755104          12.997481         8.315987
5  2023.0    2259  387945.0  5.822990           0.713330         1.179585
6  2024.0    2239  390828.0  5.728863          -0.885347        -1.616481
